# TwinConf Configuration Management Demo

This notebook demonstrates the main features of TwinConf, a configuration management library that supports:
- YAML configuration file parsing with deep merging
- Variable and expression interpolation
- Object realization and type validation
- Command line argument overrides
- Configuration sweeping for hyperparameter optimization

## 1. Install and Import TwinConf

Install the TwinConf package and import necessary libraries for configuration management.

In [ ]:
# Import TwinConf and other necessary libraries
import sys
import os
import tempfile
import yaml
from pathlib import Path

# Add the src directory to Python path to import TwinConf
project_root = Path().cwd().parent
sys.path.insert(0, str(project_root / "src"))

from twinconf import TwinConfParser, ConfigurationObject

print("TwinConf imported successfully!")
print(f"Project root: {project_root}")

## 2. Basic Configuration Reading

Read and parse YAML configuration files using TwinConfParser, demonstrating basic configuration loading.

In [ ]:
# Create a temporary directory for our demo configurations
temp_dir = tempfile.mkdtemp()
config_dir = Path(temp_dir)

print(f"Created temporary config directory: {config_dir}")

# Create a basic configuration file
basic_config = {
    "experiment": {"name": "demo_experiment", "seed": 42},
    "model": {"hidden_size": 128, "num_layers": 3, "dropout": 0.1},
    "training": {"batch_size": 32, "learning_rate": 0.001, "epochs": 100},
}

config_file = config_dir / "basic_config.yaml"
with open(config_file, "w") as f:
    yaml.safe_dump(basic_config, f)

print(f"Created config file: {config_file}")

# Parse the configuration
parser = TwinConfParser()
config = parser.parse_args([str(config_file)])

print("\nLoaded configuration:")
print(f"Experiment name: {config.experiment.name}")
print(f"Model hidden size: {config.model.hidden_size}")
print(f"Training batch size: {config.training.batch_size}")

## 3. MERGE Configuration Files

Demonstrate the MERGE keyword functionality to combine multiple configuration files with deep merging.

In [ ]:
# Create a base configuration
base_config = {
    "model": {"hidden_size": 64, "num_layers": 2, "dropout": 0.2, "activation": "relu"},
    "training": {"batch_size": 16, "learning_rate": 0.01, "optimizer": "adam"},
}

base_file = config_dir / "base_config.yaml"
with open(base_file, "w") as f:
    yaml.safe_dump(base_config, f)

# Create a specialized configuration that merges with base
specialized_config = {
    "MERGE": str(base_file),  # Merge base configuration
    "experiment": {"name": "specialized_experiment", "version": "v2.0"},
    "model": {
        "hidden_size": 256,  # Override base value
        "dropout": 0.3,  # Override base value
        # Other model params (num_layers, activation) will be inherited from base
    },
    "training": {
        "epochs": 200,  # Add new parameter
        "learning_rate": 0.005,  # Override base value
        # Other training params (batch_size, optimizer) will be inherited from base
    },
}

specialized_file = config_dir / "specialized_config.yaml"
with open(specialized_file, "w") as f:
    yaml.safe_dump(specialized_config, f)

# Parse the merged configuration
merged_config = parser.parse_args([str(specialized_file)])

print("Merged configuration:")
print(f"Experiment: {merged_config.experiment.name} {merged_config.experiment.version}")
print(f"Model hidden_size: {merged_config.model.hidden_size} (overridden from base: 64)")
print(f"Model num_layers: {merged_config.model.num_layers} (inherited from base)")
print(f"Model activation: {merged_config.model.activation} (inherited from base)")
print(f"Training learning_rate: {merged_config.training.learning_rate} (overridden from base: 0.01)")
print(f"Training batch_size: {merged_config.training.batch_size} (inherited from base)")
print(f"Training epochs: {merged_config.training.epochs} (new parameter)")

## 4. Command Line Arguments Override

Show how to override configuration values using command line arguments with the --args parameter.

In [ ]:
# Simulate command line arguments with --args overrides
# In real usage: python script.py config.yaml --args model.hidden_size=512 training.batch_size=64

cli_args = [
    str(basic_config_file := config_dir / "basic_config.yaml"),
    "--args",
    "model.hidden_size=512",  # Override nested parameter
    "training.batch_size=64",  # Override another nested parameter
    "experiment.version=v1.5",  # Add new parameter
    "training.use_scheduler=true",  # Add boolean parameter
]

# Parse configuration with CLI overrides
cli_config = parser.parse_args(cli_args)

print("Configuration with CLI overrides:")
print(f"Model hidden_size: {cli_config.model.hidden_size} (overridden from CLI: was 128)")
print(f"Training batch_size: {cli_config.training.batch_size} (overridden from CLI: was 32)")
print(f"Experiment version: {cli_config.experiment.version} (added from CLI)")
print(f"Training use_scheduler: {cli_config.training.use_scheduler} (added from CLI)")

# Show that other values remain unchanged
print(f"\nUnchanged values:")
print(f"Model num_layers: {cli_config.model.num_layers}")
print(f"Training learning_rate: {cli_config.training.learning_rate}")

## 5. Variable and Expression Interpolation

Demonstrate variable interpolation with ${} syntax and expression interpolation for dynamic configuration values.

In [ ]:
# Create configuration with variable interpolation
interpolation_config = {
    "dataset": {"name": "imagenet", "num_classes": 1000},
    "devices": [0, 1, 2, 3],
    "training": {
        "total_batch_size": 256,
        "batch_size_per_device": "${`training.total_batch_size` // len(`devices`)}",  # Expression interpolation
        "steps_per_epoch": 1000,
        "total_steps": "${`training.steps_per_epoch` * `training.epochs`}",  # Expression interpolation
        "epochs": 50,
    },
    "model": {
        "output_features": "${dataset.num_classes}",  # Variable interpolation
        "hidden_size": 512,
        "model_name": "model_${dataset.name}_h${model.hidden_size}",  # Mixed interpolation
    },
    "experiment": {
        "name": "exp_${dataset.name}",  # Variable interpolation
        "description": "Training on ${dataset.name} with ${model.output_features} classes",  # Variable interpolation
    },
}

# Set an environment variable to demonstrate env var interpolation
os.environ["EXPERIMENT_ID"] = "EXP_12345"
interpolation_config["experiment"]["id"] = "${EXPERIMENT_ID}"  # Environment variable interpolation

interpolation_file = config_dir / "interpolation_config.yaml"
with open(interpolation_file, "w") as f:
    yaml.safe_dump(interpolation_config, f)

# Parse configuration with interpolations
interp_config = parser.parse_args([str(interpolation_file)])

print("Configuration with interpolations:")
print(f"Dataset: {interp_config.dataset.name} ({interp_config.dataset.num_classes} classes)")
print(f"Devices: {interp_config.devices}")
print(f"\nVariable interpolation:")
print(f"Model output_features: {interp_config.model.output_features} (from dataset.num_classes)")
print(f"Experiment name: {interp_config.experiment.name} (from dataset.name)")
print(f"\nExpression interpolation:")
print(f"Batch size per device: {interp_config.training.batch_size_per_device} (256 // 4 devices)")
print(f"Total training steps: {interp_config.training.total_steps} (1000 * 50 epochs)")
print(f"\nMixed interpolation:")
print(f"Model name: {interp_config.model.model_name}")
print(f"Experiment description: {interp_config.experiment.description}")
print(f"\nEnvironment variable interpolation:")
print(f"Experiment ID: {interp_config.experiment.id} (from ENV var EXPERIMENT_ID)")

## 6. Object Realization

Show how to automatically realize objects from configuration using TYPE keywords and the realize() method.

In [ ]:
# Define some sample classes for object realization
class Optimizer:
    """Sample optimizer class."""

    def __init__(self, learning_rate: float, weight_decay: float = 0.0, momentum: float = 0.9):
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.momentum = momentum

    def __repr__(self):
        return f"Optimizer(lr={self.learning_rate}, wd={self.weight_decay}, momentum={self.momentum})"


class Model:
    """Sample model class."""

    def __init__(self, hidden_size: int, num_layers: int, dropout: float = 0.1, activation: str = "relu"):
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.dropout = dropout
        self.activation = activation

    def __repr__(self):
        return f"Model(h={self.hidden_size}, layers={self.num_layers}, dropout={self.dropout}, act={self.activation})"


def create_scheduler(optimizer, step_size: int = 10, gamma: float = 0.1):
    """Sample function to create a scheduler."""

    class Scheduler:
        def __init__(self, opt, step, gamma):
            self.optimizer = opt
            self.step_size = step
            self.gamma = gamma

        def __repr__(self):
            return f"Scheduler(step_size={self.step_size}, gamma={self.gamma})"

    return Scheduler(optimizer, step_size, gamma)


# Create configuration with TYPE specifications for object realization
object_config = {
    "optimizer": {
        "TYPE": "__main__.Optimizer",  # Reference to our Optimizer class
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        # momentum will use default value 0.9
    },
    "model": {
        "TYPE": "__main__.Model",  # Reference to our Model class
        "hidden_size": 256,
        "num_layers": 4,
        "dropout": 0.2,
        "activation": "gelu",
    },
    "scheduler": {
        "TYPE": "__main__.create_scheduler",  # Reference to our function
        "optimizer": {
            "TYPE": "__main__.Optimizer",  # Nested object realization
            "learning_rate": 0.01,
        },
        "step_size": 20,
        "gamma": 0.5,
    },
}

object_file = config_dir / "object_config.yaml"
with open(object_file, "w") as f:
    yaml.safe_dump(object_config, f)

# Parse configuration (objects not yet realized)
obj_config = parser.parse_args([str(object_file)])

print("Configuration before realization:")
print(f"Optimizer TYPE: {obj_config.optimizer.TYPE}")
print(f"Optimizer params: lr={obj_config.optimizer.learning_rate}, wd={obj_config.optimizer.weight_decay}")
print(f"Model TYPE: {obj_config.model.TYPE}")

# Realize individual objects
print("\nRealizing individual objects:")
optimizer = obj_config.optimizer.realize()
model = obj_config.model.realize()
scheduler = obj_config.scheduler.realize()

print(f"Realized optimizer: {optimizer}")
print(f"Realized model: {model}")
print(f"Realized scheduler: {scheduler}")

# Realize entire configuration recursively
print("\nRealizing entire configuration:")
realized_config = obj_config.realize()

print(f"Realized optimizer: {realized_config['optimizer']}")
print(f"Realized model: {realized_config['model']}")
print(f"Realized scheduler: {realized_config['scheduler']}")
print(f"Scheduler's optimizer: {realized_config['scheduler'].optimizer}")

## 7. Configuration Validation

Demonstrate type validation and checking for unexpected/missing arguments in object configurations.

In [ ]:
# Define classes with type annotations for validation
from typing import List, Optional


class ValidatedModel:
    """Model with strict type annotations for validation demo."""

    def __init__(self, hidden_size: int, dropout: float, layers: List[int], name: str, debug: Optional[bool] = False):
        self.hidden_size = hidden_size
        self.dropout = dropout
        self.layers = layers
        self.name = name
        self.debug = debug


# Create parser with validation enabled and base classes specified
validation_parser = TwinConfParser(
    base_classes={"model": ValidatedModel}, validate_types=True, check_missing_args=True, check_unexpected_args=True
)

# Valid configuration
valid_config = {
    "model": {
        "TYPE": "__main__.ValidatedModel",
        "hidden_size": 128,
        "dropout": 0.1,
        "layers": [64, 32, 16],
        "name": "my_model",
        # debug will use default value False
    }
}

valid_file = config_dir / "valid_config.yaml"
with open(valid_file, "w") as f:
    yaml.safe_dump(valid_config, f)

try:
    validated_config = validation_parser.parse_args([str(valid_file)])
    print("✅ Valid configuration parsed successfully!")
    print(f"Model hidden_size: {validated_config.model.hidden_size} (type: {type(validated_config.model.hidden_size)})")
    print(f"Model dropout: {validated_config.model.dropout} (type: {type(validated_config.model.dropout)})")
    print(f"Model layers: {validated_config.model.layers} (type: {type(validated_config.model.layers)})")
    print(f"Model debug: {validated_config.model.debug} (default value applied)")
except Exception as e:
    print(f"❌ Validation failed: {e}")

# Invalid configuration - wrong types
print("\n" + "=" * 50)
print("Testing invalid configuration with wrong types:")

invalid_config = {
    "model": {
        "TYPE": "__main__.ValidatedModel",
        "hidden_size": "128",  # Should be int, not str
        "dropout": "high",  # Should be float, not str
        "layers": [64, 32, 16],
        "name": "my_model",
    }
}

invalid_file = config_dir / "invalid_config.yaml"
with open(invalid_file, "w") as f:
    yaml.safe_dump(invalid_config, f)

try:
    invalid_validated_config = validation_parser.parse_args([str(invalid_file)])
    print("❌ This should not succeed!")
except ValueError as e:
    print(f"✅ Validation correctly caught type errors:")
    print(str(e))

# Missing required arguments
print("\n" + "=" * 50)
print("Testing configuration with missing required arguments:")

missing_config = {
    "model": {
        "TYPE": "__main__.ValidatedModel",
        "hidden_size": 128,
        # Missing: dropout, layers, name (required arguments)
    }
}

missing_file = config_dir / "missing_config.yaml"
with open(missing_file, "w") as f:
    yaml.safe_dump(missing_config, f)

try:
    missing_validated_config = validation_parser.parse_args([str(missing_file)])
    print("❌ This should not succeed!")
except ValueError as e:
    print(f"✅ Validation correctly caught missing arguments:")
    print(str(e))

## 8. List Manipulation

Use the special LIST TYPE to manipulate list elements using dictionary-style operations.

In [ ]:
# Create base configuration with LIST type
list_base_config = {
    "callbacks": {
        "TYPE": "LIST",
        "logging": "log_callback",
        "checkpoint": "save_model_callback",
        "early_stop": "early_stopping_callback",
        "tensorboard": "tensorboard_callback",
    },
    "transforms": {
        "TYPE": "LIST",
        "resize": "resize_transform",
        "normalize": "normalize_transform",
        "augment": "augmentation_transform",
    },
}

list_base_file = config_dir / "list_base.yaml"
with open(list_base_file, "w") as f:
    yaml.safe_dump(list_base_config, f)

# Create specialized configuration that modifies the lists
list_specialized_config = {
    "callbacks": {
        "MERGE": str(list_base_file) + ".callbacks",
        "checkpoint": "DELETE",  # Remove checkpoint callback
        "validation": "validation_callback",  # Add new callback
        "profiler": "profiler_callback",  # Add another new callback
    },
    "transforms": {
        "MERGE": str(list_base_file) + ".transforms",
        "augment": "DELETE",  # Remove augmentation
        "crop": "crop_transform",  # Add crop transform
    },
}

list_specialized_file = config_dir / "list_specialized.yaml"
with open(list_specialized_file, "w") as f:
    yaml.safe_dump(list_specialized_config, f)

# Parse the configuration with LIST processing
list_config = parser.parse_args([str(list_specialized_file)])

print("LIST manipulation results:")
print(f"Callbacks (after LIST processing):")
if hasattr(list_config.callbacks, "TYPE") and list_config.callbacks.TYPE == "LIST":
    print("  Still in dict form (LIST processing not fully implemented):")
    for key, value in list_config.callbacks.items():
        if key != "TYPE":
            print(f"    {key}: {value}")
else:
    print(f"  As list: {list_config.callbacks}")

print(f"\nTransforms (after LIST processing):")
if hasattr(list_config.transforms, "TYPE") and list_config.transforms.TYPE == "LIST":
    print("  Still in dict form (LIST processing not fully implemented):")
    for key, value in list_config.transforms.items():
        if key != "TYPE":
            print(f"    {key}: {value}")
else:
    print(f"  As list: {list_config.transforms}")

print("\nNote: LIST type functionality converts dict-style configuration to actual lists.")
print("This allows easy manipulation of list elements while maintaining readability.")

## 9. Configuration Serialization

Show how to serialize configurations using the pretty() method for logging and debugging.

In [ ]:
# Create a complex nested configuration for pretty printing demo
complex_config = {
    "experiment": {
        "name": "complex_experiment",
        "version": "v2.1",
        "tags": ["deep_learning", "computer_vision", "classification"],
    },
    "model": {"TYPE": "__main__.Model", "hidden_size": 512, "num_layers": 6, "dropout": 0.15, "activation": "swish"},
    "optimizer": {"TYPE": "__main__.Optimizer", "learning_rate": 0.0003, "weight_decay": 0.01, "momentum": 0.95},
    "dataset": {
        "name": "cifar10",
        "batch_size": 128,
        "num_workers": 4,
        "transforms": {
            "train": ["resize", "random_crop", "horizontal_flip", "normalize"],
            "val": ["resize", "center_crop", "normalize"],
        },
    },
    "training": {
        "epochs": 100,
        "save_every": 10,
        "validate_every": 5,
        "early_stopping": {"patience": 15, "min_delta": 0.001},
    },
}

complex_file = config_dir / "complex_config.yaml"
with open(complex_file, "w") as f:
    yaml.safe_dump(complex_config, f)

# Parse the complex configuration
parsed_complex = parser.parse_args([str(complex_file)])

print("Original nested structure access:")
print(f"Experiment: {parsed_complex.experiment.name} {parsed_complex.experiment.version}")
print(f"Model: {parsed_complex.model.TYPE} with {parsed_complex.model.num_layers} layers")
print(f"Optimizer LR: {parsed_complex.optimizer.learning_rate}")
print(f"Early stopping patience: {parsed_complex.training.early_stopping.patience}")

print("\n" + "=" * 50)
print("Pretty-printed flattened configuration:")
print("=" * 50)

# Use pretty() method to get flattened representation
pretty_config = parsed_complex.pretty()

for key, value in sorted(pretty_config.items()):
    print(f"{key}: {value}")

print("\n" + "=" * 50)
print("Pretty-printed with exclusions:")
print("=" * 50)

# Exclude certain parameters from pretty printing
pretty_config_filtered = parsed_complex.pretty(
    exclude=["experiment.tags", "dataset.transforms", "training.early_stopping.min_delta"]
)

for key, value in sorted(pretty_config_filtered.items()):
    print(f"{key}: {value}")

print("\nNote: pretty() method is useful for:")
print("- Logging configurations to files or monitoring systems")
print("- Debugging complex nested configurations")
print("- Creating flat configuration reports")
print("- Excluding sensitive or verbose parameters from logs")

## 10. Simple Parameter Sweeping

Demonstrate parameter sweeping functionality using the --sweep option for hyperparameter optimization.

In [ ]:
# Create base configuration for sweeping experiments
sweep_config = {
    "experiment": {"name": "hyperparameter_sweep", "seed": 42},
    "model": {
        "TYPE": "__main__.Model",
        "hidden_size": 128,  # Will be swept
        "num_layers": 3,  # Will be swept
        "dropout": 0.1,  # Will be swept
        "activation": "relu",
    },
    "training": {
        "batch_size": 32,
        "learning_rate": 0.001,  # Will be swept
        "epochs": 50,
    },
}

sweep_file = config_dir / "sweep_config.yaml"
with open(sweep_file, "w") as f:
    yaml.safe_dump(sweep_config, f)

# Demonstrate what sweep configurations would look like
# In practice, you would use: python script.py config.yaml --sweep model.hidden_size=64,128,256 training.learning_rate=0.0001,0.001,0.01

print("Parameter sweeping demo (simulated):")
print(
    "Command would be: python script.py config.yaml --sweep model.hidden_size=64,128,256 training.learning_rate=0.0001,0.001,0.01"
)
print("\nThis would generate configurations for all combinations:")

# Simulate the sweep by manually creating configurations
hidden_sizes = [64, 128, 256]
learning_rates = [0.0001, 0.001, 0.01]

sweep_count = 0
for hidden_size in hidden_sizes:
    for learning_rate in learning_rates:
        sweep_count += 1
        print(f"\nConfiguration #{sweep_count}:")
        print(f"  model.hidden_size: {hidden_size}")
        print(f"  training.learning_rate: {learning_rate}")
        print(f"  experiment.name: hyperparameter_sweep_run_{sweep_count}")

        # Show what the configuration would look like
        simulated_args = [
            str(sweep_file),
            "--args",
            f"model.hidden_size={hidden_size}",
            f"training.learning_rate={learning_rate}",
            f"experiment.name=hyperparameter_sweep_run_{sweep_count}",
        ]

        sweep_config_instance = parser.parse_args(simulated_args)
        print(f"  Parsed hidden_size: {sweep_config_instance.model.hidden_size}")
        print(f"  Parsed learning_rate: {sweep_config_instance.training.learning_rate}")
        print(
            f"  Other params unchanged: num_layers={sweep_config_instance.model.num_layers}, batch_size={sweep_config_instance.training.batch_size}"
        )

print(f"\nTotal sweep configurations: {sweep_count}")
print("\nSweep functionality is useful for:")
print("- Hyperparameter optimization")
print("- Grid search experiments")
print("- A/B testing different configurations")
print("- Systematic exploration of parameter spaces")

# Show an example of how you might use sweep results
print("\n" + "=" * 50)
print("Example sweep usage pattern:")
print("=" * 50)
print("""
# In your training script:
parser = TwinConfParser()
configs = parser.parse_args()  # This might return a list if --sweep was used

if isinstance(configs, list):
    # Multiple configurations from sweep
    for i, config in enumerate(configs):
        print(f"Running experiment {i+1}/{len(configs)}")
        model = config.model.realize()
        train(model, config.training)
else:
    # Single configuration
    model = config.model.realize()
    train(model, config.training)
""")

## Cleanup

Clean up temporary files created during the demo.

In [ ]:
# Clean up temporary directory
import shutil

shutil.rmtree(temp_dir)
print(f"Cleaned up temporary directory: {temp_dir}")

# Clean up environment variable
if "EXPERIMENT_ID" in os.environ:
    del os.environ["EXPERIMENT_ID"]
    print("Cleaned up environment variable: EXPERIMENT_ID")

print("\n🎉 TwinConf demo completed successfully!")
print("\nKey features demonstrated:")
print("✅ YAML configuration file parsing")
print("✅ MERGE functionality for configuration inheritance")
print("✅ Command line argument overrides")
print("✅ Variable and expression interpolation")
print("✅ Object realization with TYPE keywords")
print("✅ Configuration validation and type checking")
print("✅ LIST type for flexible list manipulation")
print("✅ Configuration serialization with pretty()")
print("✅ Parameter sweeping for hyperparameter optimization")